# ts-kit network visualization

color all nodes by location (incl. unary) 

In [4]:
def apply_styles(ts): # for 3-deme simulations. update to accept n_demes

    styles = []
    # style for each population
    for colour, p in zip(['red', 'green', 'blue'], ts.populations()): 
        # target the symbols only (class "sym")
        s = f".node.p{p.id} > .sym " + "{" + f"fill: {colour}" + "}"
        styles.append(s)
        #print(f'"{s}" applies to nodes from population {p.metadata["name"]} (id {p.id})')
    css_string = " ".join(styles)
    #print(f'CSS string applied:\n    "{css_string}"')

    colors = {"pop_0": "red", "pop_1": "green", "pop_2": "blue"}
    colors_for_node = {}
    for n in ts.samples():
        population_data = ts.population(ts.node(n).population)
        colors_for_node[n] = colors[population_data.metadata["name"]]
    
    individual_for_node = {}
    for n in ts.samples():
        individual_data = ts.individual(ts.node(n).individual)
        individual_for_node[n] = individual_data.id

    tip_lab = {u: f"{u}:p{ts.node(u).population}" for u in ts.samples()}


    return individual_for_node, css_string, tip_lab, colors_for_node


coloring branches, migration events on branches, and nodes by location

In [5]:
def apply_styles2(ts): 

    colors = ["red", "green", "blue"]
    N = 3
    
    # map population to color [red = source] 
    mig_color_dict = {}
    for i in range(N):
        rgb = plt.matplotlib.colors.to_rgb(colors[i % len(colors)])
        mig_color_dict[i] = np.array(rgb) * 255
    
    pop = ts.tables.nodes.population
    
    # must explicitly choose tree !!!!!!
    tree = ts.first()
    
    # color coalescent events, migrations, and tips 
    coal_nodes = {u for u in tree.nodes() if tree.num_children(u) >= 2}
    
    tip_nodes = {u for u in tree.nodes() if tree.num_children(u) == 0}
    
    mig_nodes = {m.node for m in ts.migrations()}
    
    show_nodes = coal_nodes | mig_nodes | tip_nodes
    
    # css
    string = ""
    string += ".tree .edge {stroke-width: 3px;}\n"
    
    # hide other stuff e.g. unary nodes
    string += ".node > .sym {opacity: 0;}\n"
    
    # edge colors
    for u in tree.nodes():
        # edge color based on population
        rgb_u = tuple(mig_color_dict[pop[u]][0:3])
        string += f".node.n{u} > .edge {{stroke: rgb{rgb_u};}}\n"
    
    for u in show_nodes:
        rgb_u = tuple(mig_color_dict[pop[u]][0:3])
        string += f".node.n{u} > .sym {{opacity: 1; fill: rgb{rgb_u};}}\n"
    
    for m in ts.migrations():
        rgb_dest = tuple(mig_color_dict[m.dest][0:3])
        string += f".node.n{m.node} > .sym {{opacity: 0.75; fill: rgb{rgb_dest};}}\n"
    
    node_labels = {node.id: "" for node in ts.nodes()}


    return string, node_labels

# quantify fraction internal nodes labeled as source

In [6]:
def source_counts(ts): 
    pop = ts.tables.nodes.population
    proportion_source = np.empty(ts.num_trees+1)
    counts_source = np.empty(ts.num_trees)
    for num, tree in enumerate(ts.trees()): 
        counts = 0
        coal_nodes = {u for u in tree.nodes() if tree.num_children(u) >= 2}
        for c in coal_nodes: 
            if pop[c] == 0: 
                counts += 1 
        proportion_source[num] = counts / len(coal_nodes)
        counts_source[num] = counts
    
    m = np.mean(proportion_source[0:ts.num_trees])
    proportion_source[ts.num_trees] = m

    return proportion_source

# quantify number of migrations per branch

In [7]:
def migrations_per_branch(ts): 

    mpb = np.empty(ts.num_trees + 1)
    mig_nodes = {m.node for m in ts.migrations()}
    for num, tree in enumerate(ts.trees()):
        tree.nodes()
        mig_counts = len(set(mig_nodes) & set(tree.nodes())) 
        mpb[num] = mig_counts / tree.num_edges
    
    m = np.mean(mpb[0:ts.num_trees])
    mpb[ts.num_trees] = m
    
    return mpb 